# Stage 4: Singlish to English Conversion

### Project Structure

```bash
CS5246Project/
    │
    ├─ data/
    │   ├─ inverted_index/
    │   │   ├─ inverted_index_tfidf_titles.json
    │   │   └─ inverted_index_tfidf_fulltext.json
    │   │
    │   ├─ models/
    │   │   ├─ sentence_bert_model/
    │   │   ├─ bm25_comments_model.joblib
    │   │   ├─ bm25_fulltext_model.joblib
    │   │   ├─ bm25_titles_model.joblib
    │   │   ├─ tfidf_comments_vectorizer.joblib
    │   │   └─ tfidf_posts_vectorizer.joblib
    │   │
    │   ├─ vector_database/
    │   │   ├─ bert_comments_embeddings.npy
    │   │   ├─ bert_contents_embeddings.npy
    │   │   ├─ bert_fulltext_embeddings.npy
    │   │   ├─ bert_titles_embeddings.npy
    │   │   ├─ bm25_comments.npz
    │   │   ├─ bm25_contents.npz
    │   │   ├─ bm25_fulltext.npz
    │   │   ├─ bm25_titles.npz
    │   │   ├─ tfidf_comments.npz
    │   │   ├─ tfidf_contents.npz
    │   │   ├─ tfidf_fulltext.npz
    │   │   └─ tfidf_titles.npz
    │   │
    │   ├─ PostVault.csv
    │   ├─ CommentVault.csv
    │   └─ raw_data/
    │
    ├─ sentiment_plots/
    │   ├─ emotion_plots/
    │   ├─ emotion_dashboard.py
    │   └─ plot_emotion_summary.py
    │
    ├─ Stage_0_Introduction.ipynb                               
    ├─ Stage_1_Data_Collection_and_Data_Cleaning.ipynb
    ├─ Stage_2_POS_and_NER_Tagging.ipynb
    ├─ Stage_3_Singlish_Normalisation.ipynb
-----------------------------------------------------------------
│   ├─ Stage_4_Singlish_to_English_Conversion.ipynb             │
-----------------------------------------------------------------            
    ├─ Stage_5_Common_Normalisation.ipynb
    ├─ Stage_6_Vector_Space_Model_and_Inverted_Index.ipynb
    ├─ Stage_7_Sentiment_Analysis.ipynb
    ├─ Stage_8_Clustering_and_Visualization.ipynb       
    ├─ Stage_9_Document_Search.ipynb
    └─ utilities/
        │
        ├─ pp_class.py
        ├─ singlish_dictionary.json
        ├─ singlish_regex_to_text.txt
        └─ slang_dictionary.csv
```

## To access the mounted folder directly.

### Step 1: Add the Shared Folder to your Google Drive (if you haven't already)

1.  **Open Google Drive** (drive.google.com) in your web browser.
2.  Go to the **'Shared with me'** section.
3.  Locate the folder that was shared with you.
4.  **Right-click** on the shared folder.
5.  Select **'Add shortcut to Drive'** (or 'Add to My Drive', depending on the interface). Choose a location within 'My Drive' if prompted, or simply add it to the root of 'My Drive'.

This creates a shortcut to the shared folder in your 'My Drive', making it accessible when Colab mounts your personal Drive.
### Step 2: Access the Shared Folder in Colab (No action needed)

Once your Drive is mounted, you can navigate to the shared folder. If you added a shortcut, it will appear in your 'My Drive' just like any other folder. The path will typically be `/content/gdrive/MyDrive/YourSharedFolderName`.

It should be able to run the script below already

In [ ]:
# -----------
# Mount Drive
# -----------
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Dependencies

### Install Dependencies

In [ ]:
!pip install emoji ftfy langdetect import-ipynb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 14.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 50.3 MB/s eta 0:00:00
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=f551273407beb1576489df47fc916bc1179fac139119cd367be6bfd25a256d23
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect


### Change Directory

In [ ]:
# Path
# ----
FOLDER_PATH: str = "/content/drive/MyDrive/CS5246Project/"

import os
os.chdir(FOLDER_PATH)

### Import Essential Library

In [ ]:
import re
import json
import import_ipynb
import pandas as pd
from tqdm import tqdm
from Stage_2_POS_and_NER_Tagging import pos_tagger
from Stage_0_Introduction import display_first_text_column_head, save_dfs_to_csv, output_base_dir, save_single_df_to_csv

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.8 MB/s eta 0:00:00
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Singlish Conversion

### Load Data

In [ ]:
# ---------------
# Load Clean Data
# ---------------
df_posts = pd.read_csv('/content/drive/MyDrive/CS5246Project/data/PostVault.csv', keep_default_na=False)
df_comments = pd.read_csv('/content/drive/MyDrive/CS5246Project/data/CommentVault.csv', keep_default_na=False)
# ------------------------
# Load Singlish Dictionary
# ------------------------
dictionary_path = '/content/drive/MyDrive/CS5246Project/utilities/singlish_dictionary.json'
with open(dictionary_path, "r") as f:
    sing_dict = json.load(f)

/tmp/ipykernel_922/3307081391.py:4: DtypeWarning: Columns (16,32) have mixed types. Specify dtype option on import or set low_memory=False.
  df_posts = pd.read_csv('/content/drive/MyDrive/CS5246Project/data/PostVault.csv', keep_default_na=False)


### Singlish to English Conversion

In [ ]:
def simple_singlish_to_english_conversion(text):
    """
    Converts Singlish phrases to English using a predefined dictionary,
    applying direct replacements without POS tagging.

    Also returns the number of conversions made.

    Args:
        text (str): The input text string which may contain Singlish.

    Returns:
        tuple: A tuple containing:
            - str: The text with Singlish terms converted to English based on the map.
            - int: The total number of conversions made.
    """
    if not isinstance(text, str):
        return "", 0

    converted_text = text
    conversion_count = 0
    # Iterate through each pattern-replacement pair in the preloaded dictionary (regex_map)
    for pattern, replacement in sing_dict.items():
        try:
            # Use re.sub to replace all occurrences of the pattern with its replacement
            # flags=re.IGNORECASE ensures the replacement is case-insensitive
            new_text, count = re.subn(
                f"\\b{pattern}\\b",
                replacement,
                converted_text,
                flags=re.IGNORECASE
            )
            if count > 0:
                converted_text = new_text
                conversion_count += count
        except re.error:
            # Catch any potential regex errors, though patterns from the file should be valid
            pass

    return converted_text, conversion_count

def convert_singlish_columns(df, text_columns):
    # Ensure 'singlish_count' column is initialized as numeric (e.g., int)
    # If it exists and is not numeric, convert it. If it doesn't exist, create it with 0.
    if 'singlish_count' not in df.columns:
        df['singlish_count'] = 0
    else:
        # Attempt to convert to numeric, coercing errors to NaN, fill NaN with 0, then cast to int.
        df['singlish_count'] = pd.to_numeric(df['singlish_count'], errors='coerce').fillna(0).astype(int)

    for col in text_columns:
        if col in df.columns:
            # Apply conversion
            conversion_results = df[col].astype(str).apply(simple_singlish_to_english_conversion)

            # Extract converted text
            df[f'english_converted_' + col.replace('singlish_normalized_', '')] = conversion_results.apply(lambda x: x[0])

            # Accumulate counts
            df['singlish_count'] += conversion_results.apply(lambda x: x[1])

    return df


In [ ]:
# ---------
# Unit Test
# ---------
print("Demonstrating simple_singlish_to_english_conversion")
print("---------------------------------------------------")
print()

sentence1 = "Wah lau, so sian today, bo liao."
sentence2 = "Can I have a packet of milo peng, please?"
sentence3 = "Dun play play, confirm plus chop very good!"

converted_s1, count_s1 = simple_singlish_to_english_conversion(sentence1)
print(f"Original: {sentence1}")
print(f"Converted: {converted_s1}")
print(f"Conversions: {count_s1}\n")

converted_s2, count_s2 = simple_singlish_to_english_conversion(sentence2)
print(f"Original: {sentence2}")
print(f"Converted: {converted_s2}")
print(f"Conversions: {count_s2}\n")

converted_s3, count_s3 = simple_singlish_to_english_conversion(sentence3)
print(f"Original: {sentence3}")
print(f"Converted: {converted_s3}")
print(f"Conversions: {count_s3}\n")

print("Note")
print("----")
print("This function performs direct replacements using the `sing_dict`.")
print("For a more robust conversion that preserves proper nouns, you should use the 'normalize_singlish_with_pos' function defined previously.")

Demonstrating simple_singlish_to_english_conversion
---------------------------------------------------

Original: Wah lau, so sian today, bo liao.
Converted: Wah lau, so bored; tired of something today, boring; nonsense; nothing better to do.
Conversions: 2

Original: Can I have a packet of milo peng, please?
Converted: Can I have a packet of milo peng, please?
Conversions: 0

Original: Dun play play, confirm plus chop very good!
Converted: Dun play play, confirm plus to stamp or seal very good!
Conversions: 1

Note
----
This function performs direct replacements using the `sing_dict`.
For a more robust conversion that preserves proper nouns, you should use the 'normalize_singlish_with_pos' function defined previously.


#### PostVault

In [ ]:
df_posts.head()

,id,title,content,score,upvote_ratio,num_comments,created_utc,subreddit_id,has_emoji,year,...,lemmatized_title,cleaned_content,singlish_normalized_content,english_converted_content,expanded_content,demojized_content,spellchecked_content,lemmatized_content,lemmatized_full_text,tfidf_cluster
0,1qs6nd8,"A Hot Hideout mala chain co-founder, 26, diagn...",,354.0,0.95,19.0,2026-01-31 16:21:49+00:00,t5_2qh8c,False,2026.0,...,fantasy meet whimsy medieval themed renaissanc...,,,,,,,,fantasy meet whimsy medieval themed renaissanc...,6.0
1,1qs5hqp,Fantasy meets whimsy at medieval-themed renais...,,153.0,0.99,15.0,2026-01-31 15:37:29+00:00,t5_2qh8c,False,2026.0,...,pair adjoining ground floor hdb shop pasir ri ...,,,,the property has a remaining tenure of 63 year...,the property has a remaining tenure of 63 year...,the property has a remaining tenure of 63 year...,property remaining tenure 63 year available sa...,pair adjoining ground floor hdb shop pasir ri ...,6.0
2,1qs39j1,Pair of adjoining ground-floor HDB shops in Pa...,the property has a remaining tenure of 63 yea...,87.0,0.89,24.0,2026-01-31 14:07:22+00:00,t5_2qh8c,False,2026.0,...,video call scam targeting elderly,the property has a remaining tenure of 63 year...,the property has a remaining tenure of 63 year...,the property has a remaining tenure of 63 year...,my 90 year old mum just got a video call from ...,my 90 year old mum just got a video call from ...,my 90 year old mum just got a video call from ...,90 year old mum got video call someone full po...,video call scam targeting elderly 90 year old ...,6.0
3,1qs1sek,Video call scam targeting elderly,My 90-year-old mum just got a video call from ...,182.0,0.96,17.0,2026-01-31 13:02:27+00:00,t5_2qh8c,False,2026.0,...,woodland checkpoint officer dragged motorist f...,my 90 year old mum just got a video call from ...,my 90 year old mum just got a video call from ...,my 90 year old mum just got a video call from ...,,,,,woodland checkpoint officer dragged motorist f...,1.0
4,1qrzbgd,Woodlands Checkpoint officer dragged by motori...,,104.0,0.92,20.0,2026-01-31 10:51:19+00:00,t5_2qh8c,False,2026.0,...,bangkok post singapore transit point uk bound ...,,,,,,,,bangkok post singapore transit point uk bound ...,1.0


In [ ]:
text_columns = ['singlish_normalized_title', 'singlish_normalized_content']
df_posts = convert_singlish_columns(df_posts, text_columns)

#### CommentVault

In [ ]:
text_columns = ['singlish_normalized_text']
df_comments = convert_singlish_columns(df_comments, text_columns)

### Save Files

In [ ]:
print("---------")
print("PostVault")
print("---------")
display(df_posts.head())
save_single_df_to_csv(df_posts, "/content/drive/MyDrive/CS5246Project/data/", "PostVault")

---------
PostVault
---------


,id,title,content,score,upvote_ratio,num_comments,created_utc,subreddit_id,has_emoji,year,...,lemmatized_title,cleaned_content,singlish_normalized_content,english_converted_content,expanded_content,demojized_content,spellchecked_content,lemmatized_content,lemmatized_full_text,tfidf_cluster
0,1qs6nd8,"A Hot Hideout mala chain co-founder, 26, diagn...",,354.0,0.95,19.0,2026-01-31 16:21:49+00:00,t5_2qh8c,False,2026.0,...,fantasy meet whimsy medieval themed renaissanc...,,,,,,,,fantasy meet whimsy medieval themed renaissanc...,6.0
1,1qs5hqp,Fantasy meets whimsy at medieval-themed renais...,,153.0,0.99,15.0,2026-01-31 15:37:29+00:00,t5_2qh8c,False,2026.0,...,pair adjoining ground floor hdb shop pasir ri ...,,,,the property has a remaining tenure of 63 year...,the property has a remaining tenure of 63 year...,the property has a remaining tenure of 63 year...,property remaining tenure 63 year available sa...,pair adjoining ground floor hdb shop pasir ri ...,6.0
2,1qs39j1,Pair of adjoining ground-floor HDB shops in Pa...,the property has a remaining tenure of 63 yea...,87.0,0.89,24.0,2026-01-31 14:07:22+00:00,t5_2qh8c,False,2026.0,...,video call scam targeting elderly,the property has a remaining tenure of 63 year...,the property has a remaining tenure of 63 year...,the property has a remaining tenure of 63 year...,my 90 year old mum just got a video call from ...,my 90 year old mum just got a video call from ...,my 90 year old mum just got a video call from ...,90 year old mum got video call someone full po...,video call scam targeting elderly 90 year old ...,6.0
3,1qs1sek,Video call scam targeting elderly,My 90-year-old mum just got a video call from ...,182.0,0.96,17.0,2026-01-31 13:02:27+00:00,t5_2qh8c,False,2026.0,...,woodland checkpoint officer dragged motorist f...,my 90 year old mum just got a video call from ...,my 90 year old mum just got a video call from ...,my 90 year old mum just got a video call from ...,,,,,woodland checkpoint officer dragged motorist f...,1.0
4,1qrzbgd,Woodlands Checkpoint officer dragged by motori...,,104.0,0.92,20.0,2026-01-31 10:51:19+00:00,t5_2qh8c,False,2026.0,...,bangkok post singapore transit point uk bound ...,,,,,,,,bangkok post singapore transit point uk bound ...,1.0


Saving DataFrames to '/content/drive/MyDrive/CS5246Project/data/' with name 'PostVault'...
----------------------------------------------------------------------------------------------------
PostVault saved to /content/drive/MyDrive/CS5246Project/data/PostVault.csv
----------------------------------------------------------------------------------------------------


In [ ]:
print("------------")
print("CommentVault")
print("------------")
display(df_comments.head())
save_single_df_to_csv(df_comments, "/content/drive/MyDrive/CS5246Project/data/", "CommentVault")